# Atera kidney: review ectopic expression with Yearbook

**Ectopic expression** is a cell expressing marker genes of a different cell
type. It can come from transcripts of a neighboring cell assigned to the wrong
cell (spillover), from two cells segmented as one, or from real biology, and
the expression values alone can't tell these apart. Looking at where the
transcripts sit in the tissue often can. This notebook finds cells with strong
ectopic expression using `dega.select`, ranks them, and reviews them in a
**Yearbook** with one marker of the cell's own type and the ectopic gene
highlighted.

**Why the glomerulus?** Podocytes wrap their foot processes around the
glomerular capillaries, so podocytes and glomerular endothelial cells sit right
next to each other. Podocin (NPHS2) is specific to podocytes, and endomucin
(EMCN) marks the endothelial cells. Podocin transcripts in an endothelial cell
are ectopic, and the shared border with a podocyte makes spillover a likely
cause. The companion [Yearbook review](Atera_Kidney_Yearbook_Review.ipynb)
looks at the same cells from another angle: large cells that may be two cells
segmented as one.

The data are the public [10x Genomics Atera WTA FFPE human non-diseased kidney dataset](https://www.10xgenomics.com/datasets/atera-wta-ffpe-human-nd-kidney), using the same [prepared DegaFiles](https://github.com/broadinstitute/Celldega_Atera_v1_WTA_Kidney_non_diseased_FFPE) as the [Atera kidney tutorial](../tutorial_notebooks/Atera_Kidney_Landscape-Clustergram.ipynb). The notebook downloads only what it needs (cell IDs, cluster labels, and the expression of two genes, about 10 MB); the Yearbook streams images and cell boundaries as you page.

This example requires Celldega 0.29.1 or later: `pip install "celldega>=0.29.1"`.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve

import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse

import celldega as dega

In [2]:
SAMPLE = "Atera_v1_WTA_Kidney_non_diseased_FFPE"
DATA_REVISION = "3a46ab4db67821eef3b8ff8d1c3704fb7b3bb48f"
BASE_URL = f"https://raw.githubusercontent.com/broadinstitute/Celldega_{SAMPLE}/{DATA_REVISION}"
ANNDATA_URL = (
    "https://huggingface.co/datasets/broadinstitute/Celldega_Supporting_Data/"
    f"resolve/main/{SAMPLE}.h5ad"
)

cwd = Path.cwd().resolve()
repo = next((p for p in [cwd, *cwd.parents] if (p / "pyproject.toml").is_file()), None)
cache = (repo or cwd) / "data/atera_kidney_review"
cache.mkdir(parents=True, exist_ok=True)


def fetch(relative):
    """Download one DegaFiles file into the cache, once."""
    path = cache / relative
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        urlretrieve(f"{BASE_URL}/{relative}", path)
    return path


cell_ids = pd.Index(
    pd.read_parquet(fetch("cell_metadata.parquet"), columns=["name"])["name"].astype(str),
    name="cell_id",
)
f"{len(cell_ids):,} cells"

'399,120 cells'

## 1. Load the clusters and two marker genes

Build a small `AnnData` with NPHS2 and EMCN counts and the default cluster
labels for every cell. In this dataset's default clusters, cluster 16 contains
podocytes and cluster 20 glomerular endothelial cells. The DegaFiles store each
gene's expression as a sparse column indexed by cell position.

In [3]:
GENES = ["NPHS2", "EMCN"]  # podocyte marker, endothelial marker
columns = []
for gene in GENES:
    frame = pd.read_parquet(fetch(f"cbg/{gene}.parquet"))
    rows = frame.index.to_numpy(dtype=np.int64)
    columns.append(sparse.csc_matrix(
        (frame[gene].to_numpy(), (rows, np.zeros(len(rows), dtype=np.int64))),
        shape=(len(cell_ids), 1),
    ))

cells = ad.AnnData(
    X=sparse.hstack(columns, format="csr"),
    obs=pd.DataFrame(index=cell_ids),
    var=pd.DataFrame(index=GENES),
)
clusters = pd.read_parquet(fetch("cell_clusters/cluster.parquet"))["cluster"]
cells.obs["cluster"] = clusters.astype(str).reindex(cells.obs_names)
cells

AnnData object with n_obs × n_vars = 399120 × 2
    obs: 'cluster'

## 2. Find where the unexpected gene is detected

The share of cells in each cluster with at least 2 NPHS2 counts shows where
podocin appears. Podocytes (cluster 16) come first, as expected. Glomerular
endothelial cells (cluster 20) come next by a wide margin, while podocin is
almost absent from every other cluster.

In [4]:
nphs2 = cells[:, "NPHS2"].X.toarray().ravel()
detected = (
    pd.Series(nphs2 >= 2, index=cells.obs_names)
    .groupby(cells.obs["cluster"])
    .agg(["mean", "size"])
    .rename(columns={"mean": "share with NPHS2 >= 2", "size": "cells"})
    .sort_values("share with NPHS2 >= 2", ascending=False)
)
detected.head(5).style.format({"share with NPHS2 >= 2": "{:.1%}", "cells": "{:,}"})

,share with NPHS2 >= 2,cells
cluster,,
16,87.1%,"5,172"
20,26.5%,"4,118"
5,6.0%,"9,353"
7,0.5%,"11,178"
0,0.1%,"14,211"


## 3. Select the strongest ectopic expression

Query the glomerular endothelial cells with at least 2 NPHS2 counts, then rank
them by NPHS2, highest first. The top of the ranking is where ectopic
expression is strongest and easiest to judge by eye.

In [5]:
ENDOTHELIAL = "20"

selector = dega.select.Selector(cells)
ectopic = (selector.attr("cluster") == ENDOTHELIAL) & (selector.gene("NPHS2") >= 2)
review = selector.select(
    query=ectopic,
    sampler=selector.samplers.rank(attr=selector.gene("NPHS2"), n=30, by="high"),
)
print(f"{review.candidate_count:,} endothelial cells with NPHS2 >= 2; "
      f"reviewing the {len(review)} with the most.")
review.to_dataframe().head()

1,091 endothelial cells with NPHS2 >= 2; reviewing the 30 with the most.


,id,rank,score
0,jmdgohce-1,0,38.0
1,djlmkakh-1,1,30.0
2,fogmeeaf-1,2,27.0
3,kbdlhlek-1,3,25.0
4,ejnfnnjm-1,4,23.0


## 4. Review them in a Yearbook

`genes=["EMCN", "NPHS2"]` shows only those genes' transcripts, each in its own
color: the cell's own marker and the ectopic gene. Look at where the NPHS2
transcripts sit:

- **Along the border with a neighboring podocyte:** likely spillover.
- **Throughout a boundary that seems to enclose two nuclei:** likely two cells
  segmented as one.
- **Spread through an endothelial cell, away from any podocyte:** not explained
  by a neighbor; worth a closer look.

Check the box at the lower right of each portrait that does **not** look like
spillover.

In [6]:
yearbook = dega.viz.Yearbook(
    name="kidney_ectopic_review", base_url=BASE_URL, selection=review,
    genes=["EMCN", "NPHS2"], rows=3, cols=5, portrait_size_um=40,
    dataset_name="Atera kidney: NPHS2 in glomerular endothelial cells",
    width=0, height=850,
)
yearbook.layout.width = "100%"

In [7]:
yearbook

## 5. Summarize the review

`yearbook.selected_cells` returns the cells you flagged as not spillover. The
rest support treating podocin in these endothelial cells as spillover, for
example by not interpreting it as endothelial expression, or by assigning the
transcripts to the neighboring podocytes if your segmentation tool allows it.

In [8]:
flagged = yearbook.selected_cells
if flagged:
    print(f"{len(review) - len(flagged)} of {len(review)} reviewed cells look like spillover; "
          f"{len(flagged)} do not.")
else:
    print("No cells flagged yet: check portraits in a live notebook, then rerun this cell.")

No cells flagged yet: check portraits in a live notebook, then rerun this cell.
